In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [3]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [4]:
loss, mae = model.evaluate(X_test, y_test, verbose=0)

print("Test MSE:", loss)
print("Test MAE:", mae)

Test MSE: 5.2674126625061035
Test MAE: 1.991317629814148


In [5]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(y_train.reshape(-1, 1))
y_test = encoder.transform(y_test.reshape(-1, 1))

In [6]:
model = keras.Sequential([
    keras.layers.Input(shape=(4,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 58ms/step - accuracy: 0.6354 - loss: 0.8955 - val_accuracy: 0.5833 - val_loss: 0.8451
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6979 - loss: 0.7825 - val_accuracy: 0.5833 - val_loss: 0.7694
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7396 - loss: 0.7104 - val_accuracy: 0.5833 - val_loss: 0.7066
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7396 - loss: 0.6751 - val_accuracy: 0.5833 - val_loss: 0.6570
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8021 - loss: 0.6061 - val_accuracy: 0.5833 - val_loss: 0.6153
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7917 - loss: 0.5457 - val_accuracy: 0.6250 - val_loss: 0.5811
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7917 - loss: 0.5032 - val_accuracy: 0.6250 - val_loss: 0.5512
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8229 - loss: 0.4676 - val_accuracy: 0.6667 - val_loss: 0.5235


In [8]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

Test Loss: 0.3359820544719696
Test Accuracy: 0.7666666507720947


In [9]:
!pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.6 MB/s eta 0:00:00


In [10]:
import keras_tuner as kt

In [11]:
def build_model(hp):
    model = keras.Sequential()

    model.add(keras.layers.Input(shape=(4,)))

    for i in range(hp.Int("layers", 1, 3)):
        model.add(keras.layers.Dense(
            hp.Int("units", 16, 64, step=16),
            activation="relu"
        ))
        model.add(keras.layers.Dropout(
            hp.Float("dropout", 0.0, 0.5, step=0.1)
        ))

    model.add(keras.layers.Dense(3, activation="softmax"))

    optimizer = hp.Choice("optimizer", ["adam", "sgd"])

    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [12]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    directory="tuner",
    project_name="iris_mlp"
)

tuner.search(
    X_train, y_train,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.5833333134651184

Best val_accuracy So Far: 0.8333333134651184
Total elapsed time: 00h 00m 19s


In [13]:
best_model = tuner.get_best_models(num_models=1)[0]

loss, accuracy = best_model.evaluate(
    X_test, y_test, verbose=0
)

print("Best Model Accuracy:", accuracy)
print("Best Hyperparameters:")
print(tuner.get_best_hyperparameters(1)[0].values)

Best Model Accuracy: 0.800000011920929
Best Hyperparameters:
{'layers': 1, 'units': 48, 'dropout': 0.1, 'optimizer': 'sgd'}
